# Khảo sát và tiền xử lý dữ liệu Bank Marketing

Notebook này dùng bộ **Bank Marketing bank-additional-full** của UCI (41.188 lượt liên hệ, 20 biến đầu vào và nhãn y). Các bước gồm đọc dữ liệu, khảo sát cấu trúc và chất lượng dữ liệu, phân tích các thuộc tính trong yêu cầu, làm sạch, mã hóa và xuất dữ liệu.

> Bộ dữ liệu được tải tự động từ UCI nếu thư mục chạy chưa có tệp CSV. Trên Google Colab, chọn **Runtime → Run all** để chạy lần lượt toàn bộ notebook.

Nguồn: [UCI Machine Learning Repository — Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank%2Bmarketing).


## 1. Thư viện và đọc dữ liệu

Dùng bank-additional-full.csv vì bộ dữ liệu này có các thuộc tính duration và cons.price.idx như trong yêu cầu. Dữ liệu gốc phân tách bằng dấu chấm phẩy.


In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

local_candidates = [Path("bank-additional-full.csv"), Path("bank_marketing.csv")]
local_csv = next((path for path in local_candidates if path.exists()), None)

if local_csv is not None:
    df = pd.read_csv(local_csv, sep=";")
    print(f"Đọc dữ liệu từ: {local_csv.resolve()}")
else:
    url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
    with urllib.request.urlopen(url, timeout=60) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        csv_member = next(name for name in archive.namelist() if name.endswith("bank-additional-full.csv"))
        with archive.open(csv_member) as csv_file:
            df = pd.read_csv(csv_file, sep=";")
    print(f"Đã tải dữ liệu UCI: {csv_member}")

df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={"y": "subscribe"})
print("Kích thước dữ liệu:", df.shape)
display(df.head())


## 2. Mô tả các thuộc tính

| Thuộc tính | Ý nghĩa |
|---|---|
| job | Nghề nghiệp |
| marital | Tình trạng hôn nhân |
| education | Trình độ học vấn |
| default | Có tín dụng mặc định hay không |
| loan | Có khoản vay cá nhân hay không |
| duration | Thời lượng lần liên hệ cuối, tính bằng giây |
| cons.price.idx | Chỉ số giá tiêu dùng |
| subscribe | Có đăng ký tiền gửi có kỳ hạn hay không (yes/no), đổi tên từ y |

Bộ dữ liệu còn có tuổi, khoản vay mua nhà, kênh và ngày liên hệ, số lần liên hệ, kết quả chiến dịch trước đó và một số chỉ số kinh tế. duration chỉ được biết sau cuộc gọi; cần cân nhắc bỏ thuộc tính này nếu dùng dữ liệu để dự đoán trước khi gọi.


## 3. Cấu trúc và thống kê tổng quan


In [ ]:
print("Thông tin kiểu dữ liệu và số giá trị không rỗng:")
df.info()
print("\nThống kê các biến số:")
display(df.describe().T.round(2))
print("\nThống kê các biến phân loại:")
display(df.describe(include="object").T)
print("\nSố dòng trùng hoàn toàn:", int(df.duplicated().sum()))


## 4. Giá trị thiếu và giá trị unknown

Trong dữ liệu UCI, nhiều trường chưa biết được ghi là chuỗi unknown thay vì ô trống. Thống kê cả hai loại để thấy rõ mức độ thiếu thông tin trước khi xử lý.


In [ ]:
missing_report = pd.DataFrame({
    "so_na": df.isna().sum(),
    "ti_le_na_%": (df.isna().mean() * 100).round(2),
    "so_unknown": df.astype("string").apply(lambda col: col.str.strip().str.lower().eq("unknown").sum()),
})
missing_report["ti_le_unknown_%"] = (missing_report["so_unknown"] / len(df) * 100).round(2)
display(missing_report.sort_values(["so_unknown", "so_na"], ascending=False))


## 5. Phân bố nhãn subscribe và các thuộc tính phân loại


In [ ]:
print("Tần suất nhãn subscribe:")
display(df["subscribe"].value_counts(dropna=False).rename_axis("subscribe").to_frame("so_luong"))
print("Tỉ lệ nhãn (%):")
display((df["subscribe"].value_counts(normalize=True) * 100).round(2).rename("ti_le_%").to_frame())

categorical_cols = df.select_dtypes(include="object").columns.tolist()
for col in categorical_cols:
    print(f"\nPhân bố {col}:")
    display(df[col].value_counts(dropna=False).head(15).to_frame("so_luong"))


## 6. Khảo sát các thuộc tính trong yêu cầu

Biểu đồ cho thấy phân bố nghề nghiệp, tình trạng hôn nhân, học vấn, tín dụng mặc định và khoản vay. Với các nhóm có giá trị unknown, giữ chúng trong phần khảo sát để không che khuất mức độ thiếu thông tin ban đầu.


In [ ]:
focus_categorical = ["job", "marital", "education", "default", "loan"]
fig, axes = plt.subplots(3, 2, figsize=(16, 15))
for ax, col in zip(axes.flat, focus_categorical):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, color="#4c72b0")
    ax.set_title(f"Phân bố {col}")
    ax.set_xlabel("Số lượt liên hệ")
    ax.set_ylabel(col)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()


### 6.1 Thời lượng liên hệ và chỉ số giá tiêu dùng

duration thường lệch phải mạnh; xem cả thống kê phân vị và biểu đồ sau khi giới hạn trục ở phân vị 99 để những giá trị rất lớn không làm khó đọc phân bố. cons.price.idx là chỉ số kinh tế vĩ mô, nên thường có ít mức giá trị hơn các biến khách hàng.


In [ ]:
for col in ["duration", "cons.price.idx"]:
    print(f"Thống kê {col}:")
    display(df[col].describe(percentiles=[.25, .5, .75, .9, .95, .99]).to_frame().T.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
duration_cap = df["duration"].quantile(.99)
sns.histplot(df.loc[df["duration"] <= duration_cap, "duration"], bins=40, kde=True, ax=axes[0], color="#4c72b0")
axes[0].set_title("Phân bố duration (đến phân vị 99)")
axes[0].set_xlabel("duration (giây)")
sns.histplot(df["cons.price.idx"], bins=30, ax=axes[1], color="#55a868")
axes[1].set_title("Phân bố cons.price.idx")
axes[1].set_xlabel("cons.price.idx")
plt.tight_layout()
plt.show()


## 7. Mối quan hệ giữa các thuộc tính và kết quả đăng ký


In [ ]:
for col in focus_categorical:
    rate = (df.groupby(col, dropna=False)["subscribe"]
              .apply(lambda values: values.eq("yes").mean())
              .sort_values(ascending=False))
    fig, ax = plt.subplots(figsize=(9, max(3.5, .38 * len(rate))))
    sns.barplot(x=rate.values, y=rate.index.astype(str), ax=ax, color="#55a868")
    ax.set_title(f"Tỉ lệ subscribe=yes theo {col}")
    ax.set_xlabel("Tỉ lệ đăng ký")
    ax.set_ylabel(col)
    ax.set_xlim(0, max(.1, rate.max() * 1.15))
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1%", padding=3)
    plt.tight_layout()
    plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=df, x="subscribe", y="duration", showfliers=False, ax=axes[0])
axes[0].set_title("duration theo subscribe (ẩn điểm ngoại lai)")
axes[0].set_xlabel("subscribe")
axes[0].set_ylabel("duration (giây)")
sns.boxplot(data=df, x="subscribe", y="cons.price.idx", showfliers=True, ax=axes[1])
axes[1].set_title("cons.price.idx theo subscribe")
axes[1].set_xlabel("subscribe")
axes[1].set_ylabel("cons.price.idx")
plt.tight_layout()
plt.show()

print("Tỉ lệ subscribe=yes:", f"{df['subscribe'].eq('yes').mean():.2%}")
print("Thời lượng trung vị theo nhãn:")
display(df.groupby("subscribe")["duration"].median().to_frame("duration_trung_vi"))


## 8. Nhận xét khảo sát

Các kết quả chi tiết được in ngay phía trên khi chạy notebook. Chú ý nhãn subscribe thường mất cân bằng (số no nhiều hơn yes), các trường unknown là thiếu thông tin có chủ đích trong dữ liệu, và duration có thể gây rò rỉ thông tin nếu mục tiêu là dự đoán trước cuộc gọi. Không nên diễn giải tương quan quan sát được như quan hệ nhân quả.


## 9. Tiền xử lý dữ liệu

Quy tắc áp dụng:
- Chuẩn hóa tên trường thành chữ thường; đổi y thành subscribe.
- Loại bỏ dòng trùng hoàn toàn.
- Chuẩn hóa chuỗi rỗng thành giá trị thiếu; giữ nhãn unknown như một nhóm riêng vì đây là thông tin về trạng thái chưa biết.
- Điền thiếu ở biến số bằng trung vị; biến phân loại bằng unknown.
- Giữ dữ liệu phân loại dạng chữ trong bản sạch, đổi subscribe thành 0/1 trong bản mã hóa.
- Tạo bản mã hóa one-hot; bỏ duration khỏi bản dùng cho dự đoán trước cuộc gọi để tránh rò rỉ thông tin. Bản đầy đủ đã làm sạch vẫn giữ duration cho khảo sát.


In [ ]:
clean = df.copy()
clean.columns = clean.columns.str.strip().str.lower()
clean = clean.drop_duplicates().copy()

for col in clean.select_dtypes(include="object").columns:
    clean[col] = clean[col].astype("string").str.strip()
    clean[col] = clean[col].replace("", pd.NA)

numeric_cols = clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = clean.select_dtypes(include=["object", "string", "category"]).columns.tolist()
for col in numeric_cols:
    clean[col] = clean[col].fillna(clean[col].median())
for col in categorical_cols:
    clean[col] = clean[col].fillna("unknown").astype(str).str.lower()

clean["subscribe"] = clean["subscribe"].str.lower().map({"no": 0, "yes": 1}).astype("int8")
print("Kích thước sau xử lý:", clean.shape)
print("Tổng giá trị thiếu còn lại:", int(clean.isna().sum().sum()))
print("Tên cột đã viết thường:", all(col == col.lower() for col in clean.columns))
print("Nhãn subscribe (0=no, 1=yes):")
display(clean["subscribe"].value_counts().sort_index().rename_axis("subscribe").to_frame("so_luong"))
display(clean.head())


## 10. Kiểm tra và mã hóa dữ liệu


In [ ]:
target = clean["subscribe"]
features = clean.drop(columns=["subscribe", "duration"], errors="ignore")
cat_features = features.select_dtypes(include="object").columns.tolist()
num_features = features.select_dtypes(include=np.number).columns.tolist()

encoded_features = pd.get_dummies(features, columns=cat_features, dummy_na=False, dtype="int8")
encoded_features.columns = encoded_features.columns.str.lower()
encoded = encoded_features.copy()
encoded["subscribe"] = target.astype("int8")

print("Số biến số ban đầu:", len(num_features))
print("Số biến phân loại được mã hóa:", len(cat_features))
print("Kích thước bảng đã mã hóa:", encoded.shape)
print("Còn giá trị thiếu:", int(encoded.isna().sum().sum()))
print("Tất cả tên cột chữ thường:", all(col == col.lower() for col in encoded.columns))
display(encoded.head())


## 11. Xuất các tệp kết quả

bank_marketing_clean.csv giữ các trường phân loại ở dạng chữ và có duration; bank_marketing_encoded.csv là bảng số đã one-hot, nhãn subscribe mã hóa 0/1 và không có duration để tránh rò rỉ khi dự đoán trước cuộc gọi. Trên Colab, hai tệp xuất hiện trong thư mục làm việc hiện tại.


In [ ]:
clean_path = Path("bank_marketing_clean.csv")
encoded_path = Path("bank_marketing_encoded.csv")
clean.to_csv(clean_path, index=False)
encoded.to_csv(encoded_path, index=False)
print(f"Đã lưu: {clean_path.resolve()} ({clean.shape[0]} dòng, {clean.shape[1]} cột)")
print(f"Đã lưu: {encoded_path.resolve()} ({encoded.shape[0]} dòng, {encoded.shape[1]} cột)")


## Kết luận

Notebook đã khảo sát dữ liệu và các thuộc tính được yêu cầu, thống kê missing/unknown, trực quan hóa phân bố và tỉ lệ đăng ký, làm sạch dữ liệu, mã hóa biến phân loại và xuất hai CSV. Khi diễn giải kết quả, lưu ý dữ liệu chỉ phản ánh chiến dịch của một ngân hàng tại Bồ Đào Nha và không chứng minh quan hệ nhân quả.
